# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the **FAIR²** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is exposed as a [Croissant schema](https://mlcommons.org/croissant/), allowing structured exploration, filtering, and transformation in a reproducible way. All entities (record sets, fields, columns) are referenced by their `@id`.

### Dataset Source
The dataset Croissant schema is available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install mlcroissant if not installed
!pip install mlcroissant==1.0.22

## 1. Data Loading

Load Croissant metadata and prepare for records exploration. Every record set and field is referenced by its `@id` for reproducibility.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object (not as dict/list)
metadata = dataset.metadata  # This is a Metadata object
print(f"Dataset loaded: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Explore available record sets and their fields, all referenced by `@id`s. This helps identify what tabular structures and attributes are available for analysis.

We print out record set `@id`s and for each one, the available field/column `@id`s and their names.

In [ ]:
# List all record sets in the dataset
record_sets = []
if hasattr(metadata, "record_sets"):
    for rs in metadata.record_sets:
        print(f"RecordSet: @id={rs['@id']}, name: {rs.get('name', '(no name)')}")
        record_sets.append(rs['@id'])
        # List fields/columns in each record set, if present
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"    Field: @id={field['@id']}, name: {field.get('name', '(no name)')}, dataType: {field.get('dataType', '(unknown)')}")
        elif 'columns' in rs:
            for col in rs['columns']:
                print(f"    Column: @id={col['@id']}, name: {col.get('name', '(no name)')}, dataType: {col.get('dataType', '(unknown)')}")
        print()
else:
    # Older Croissant schemas may use metadata.datasets or attach record sets elsewhere
    print("No record sets discovered in metadata.")

## 3. Data Extraction

Let's extract tabular data from one or more of the available record sets (`@id` values from above). Data is loaded into Pandas DataFrames, with column names corresponding to field/column `@id`s.

In [ ]:
# - Replace with the record set @id found from previous cell
# Example: 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-records'

RECORD_SET_IDS = []

# Try to automatically discover record set IDs
if hasattr(metadata, "record_sets"):
    RECORD_SET_IDS = [rs["@id"] for rs in metadata.record_sets]
else:
    print("No record sets available for data extraction.")

# Load each record set into a separate DataFrame
dataframes = {}
for rs_id in RECORD_SET_IDS:
    print(f"Loading record set: {rs_id}")
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"  Columns ({len(df.columns)}): {df.columns.tolist()}")
    print(f"  Sample:\n{df.head()}\n")

# For analysis, pick the main/first record set
if RECORD_SET_IDS:
    MAIN_RECORD_SET_ID = RECORD_SET_IDS[0]
    print(f"Primary record set chosen: {MAIN_RECORD_SET_ID}")
else:
    MAIN_RECORD_SET_ID = None


## 4. Exploratory Data Analysis (EDA)

We demonstrate common data processing steps:
- **Filtering**: Select only rows where a numeric field exceeds a value.
- **Normalization**: Standardize a numeric column.
- **Grouping**: Aggregate data by a categorical column.

For operations below, reference field/column `@id`s exactly.

In [ ]:
import numpy as np

if MAIN_RECORD_SET_ID and MAIN_RECORD_SET_ID in dataframes:
    df = dataframes[MAIN_RECORD_SET_ID].copy()
    # List numeric field candidates
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Detected numeric fields: {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Reference by @id
        threshold = df[numeric_field].quantile(0.75)  # Use 3rd quartile as a filter threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Records where {numeric_field} > {threshold:.2f} (75th percentile): {len(filtered_df)} rows")
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Grouping: try a non-numeric field
        non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
        if non_numeric_fields:
            group_field = non_numeric_fields[0]   # Use first non-numeric field
            print(f"Grouping by {group_field} (if high cardinality, output only first few groups)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print('No categorical field available for grouping.')
    else:
        print("No numeric fields found in main record set - cannot perform EDA.")
else:
    print('No main DataFrame loaded for EDA.')

## 5. Visualization

Visualize distributions or relationships between key dataset fields. For example, plot a histogram for a main numeric field, or a boxplot grouped by a categorical attribute.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if MAIN_RECORD_SET_ID and MAIN_RECORD_SET_ID in dataframes:
    df = dataframes[MAIN_RECORD_SET_ID]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(7,4))
        df[numeric_field].hist(bins=20)
        plt.title(f"Histogram of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
        # Boxplot by a category
        non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
        if non_numeric_fields:
            group_field = non_numeric_fields[0]
            plt.figure(figsize=(10,4))
            df.boxplot(column=numeric_field, by=group_field)
            plt.title(f"Boxplot of {numeric_field} by {group_field}")
            plt.suptitle("")
            plt.show()
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion

In this notebook, we've:
- Loaded Croissant metadata from the FAIR² dataset
- Programmatically explored available record sets and their fields by `@id`
- Loaded tabular data into Pandas and performed initial filtering, normalization, and grouping
- Visualized the numeric distributions, facilitating deeper analysis

All exploration references record set and field `@id`s to ensure clarity and reproducibility. Further analysis can be performed on any record set or field using the mlcroissant library as demonstrated.